# H&M LightGBM Baseline: Ablation Study and SHAP Analysis

This notebook completes two analyses for the current H&M LightGBM recommendation baseline:

1. Top-5 feature ablation: remove each high-gain feature and compare MAP@12.
2. SHAP analysis: generate a global beeswarm plot and one local waterfall plot.

Run this notebook on Kaggle, with the H&M competition dataset and the project code dataset attached.

## 1. Environment Setup

The code below locates the uploaded project scripts, copies them into `/kaggle/working`, and locates the H&M competition data.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

WORK_DIR = Path('/kaggle/working')
INPUT_DIR = Path('/kaggle/input')
WORK_DIR.mkdir(parents=True, exist_ok=True)

for filename in ['Final_Project.py', 'Final_Project_LGBM.py']:
    matches = list(INPUT_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(f'Cannot find {filename}. Attach the code dataset that contains this file.')
    shutil.copy(matches[0], WORK_DIR / filename)
    print(f'copied {matches[0]} -> {WORK_DIR / filename}')

data_candidates = [
    Path('/kaggle/input/h-and-m-personalized-fashion-recommendations'),
    Path('/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations'),
]
required_files = ['articles.csv', 'customers.csv', 'transactions_train.csv', 'sample_submission.csv']
for candidate in data_candidates:
    if all((candidate / name).exists() for name in required_files):
        os.environ['HNM_DATA_PATH'] = str(candidate)
        break
else:
    raise FileNotFoundError('Cannot locate the H&M competition data under /kaggle/input.')

os.chdir(WORK_DIR)
print('HNM_DATA_PATH =', os.environ['HNM_DATA_PATH'])
print('cwd =', Path.cwd())

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm', 'shap'])
print('dependencies ready')

## 2. Load Data and Build Validation Split

To keep the analysis practical on Kaggle CPU, the notebook uses capped customer samples for ablation and SHAP. Increase the caps if you have more time.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import shap

from Final_Project import (
    BASE_PATH,
    DEFAULT_RANKER,
    MAX_K,
    OUTPUT_DIR,
    fit_recommender,
    load_ranker_from_cache,
    load_tables,
    prepare_article_department,
    prepare_customer_age_bin,
    prepare_output_dir,
    prepare_transactions,
    resolve_base_path,
)
from Final_Project_LGBM import (
    FEATURE_COLUMNS,
    MAX_CANDIDATES_PER_CUSTOMER,
    RANDOM_STATE,
    _collect_actual_items,
    build_candidate_frame,
    build_feature_frame,
    build_labeled_window_dataset,
    build_train_and_validation_windows,
    downsample_training_rows,
    mapk12,
    prepare_article_model_features,
    scored_frame_to_prediction_map,
)

OUTPUT_DIR = prepare_output_dir(OUTPUT_DIR)

ANALYSIS_TRAIN_CUSTOMER_CAP = 30000
ANALYSIS_VALID_CUSTOMER_CAP = 30000
SHAP_SAMPLE_SIZE = 3000
N_ESTIMATORS = 260

base_path = resolve_base_path(BASE_PATH)
tables = load_tables(base_path)
transactions = prepare_transactions(tables['transactions'])
article_department = prepare_article_department(tables['articles'])
article_features = prepare_article_model_features(tables['articles'])
customer_age_bin = prepare_customer_age_bin(tables['customers'])
ranker_config = load_ranker_from_cache(output_dir=OUTPUT_DIR, fallback=DEFAULT_RANKER)

train_window, validation_window = build_train_and_validation_windows(transactions)
print('train_window =', train_window)
print('validation_window =', validation_window)
print('feature_count =', len(FEATURE_COLUMNS))

In [ ]:
train_df_raw = build_labeled_window_dataset(
    transactions=transactions,
    article_department=article_department,
    article_features=article_features,
    customer_age_bin=customer_age_bin,
    ranker_config=ranker_config,
    window=train_window,
    customer_cap=ANALYSIS_TRAIN_CUSTOMER_CAP,
)
train_df = downsample_training_rows(train_df_raw)
print('train rows after downsample:', train_df.height)
print(train_df.select(pl.col('label').sum().alias('positives'), pl.len().alias('rows')))

In [ ]:
history_tx = transactions.filter(pl.col('t_dat') <= pl.lit(validation_window.history_end))
label_tx = transactions.filter(
    (pl.col('t_dat') >= pl.lit(validation_window.label_start))
    & (pl.col('t_dat') <= pl.lit(validation_window.label_end))
)
valid_customers, actual_map = _collect_actual_items(label_tx, customer_cap=ANALYSIS_VALID_CUSTOMER_CAP)
valid_artifacts = fit_recommender(
    history_tx,
    article_department=article_department,
    customer_age_bin=customer_age_bin,
)
valid_candidates = build_candidate_frame(
    customer_ids=valid_customers,
    artifacts=valid_artifacts,
    ranker_config=ranker_config,
    max_candidates=MAX_CANDIDATES_PER_CUSTOMER,
)
valid_features = build_feature_frame(
    candidates=valid_candidates,
    history_tx=history_tx,
    article_features=article_features,
    customer_age_bin=customer_age_bin,
    artifacts=valid_artifacts,
    reference_date=validation_window.history_end,
)
actual_list = [actual_map.get(customer_id, []) for customer_id in valid_customers]
print('validation customers:', len(valid_customers))
print('validation candidate rows:', valid_features.height)

## 3. Train Baseline Model and Select Top-5 Gain Features

In [ ]:
def train_model(train_frame: pl.DataFrame, feature_cols: list[str], n_estimators: int = N_ESTIMATORS):
    x_train = train_frame.select(feature_cols).to_pandas()
    y_train = train_frame.get_column('label').to_numpy()
    positive_count = float(np.sum(y_train))
    negative_count = float(len(y_train) - positive_count)
    scale_pos_weight = max(1.0, negative_count / max(positive_count, 1.0))
    model = lgb.LGBMClassifier(
        objective='binary',
        boosting_type='gbdt',
        n_estimators=n_estimators,
        learning_rate=0.045,
        num_leaves=96,
        min_child_samples=80,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        force_col_wise=True,
        verbose=-1,
    )
    model.fit(x_train, y_train, feature_name=feature_cols)
    return model

def evaluate_map12(model, feature_frame: pl.DataFrame, feature_cols: list[str]) -> float:
    x_valid = feature_frame.select(feature_cols).to_pandas()
    scores = model.predict_proba(x_valid)[:, 1]
    scored = feature_frame.with_columns(pl.Series('score', scores))
    pred_map = scored_frame_to_prediction_map(scored, k=MAX_K)
    predicted_list = [pred_map.get(customer_id, []) for customer_id in valid_customers]
    return mapk12(actual_list, predicted_list, k=MAX_K)

baseline_model = train_model(train_df, FEATURE_COLUMNS)
baseline_map12 = evaluate_map12(baseline_model, valid_features, FEATURE_COLUMNS)
print(f'Baseline MAP@12 = {baseline_map12:.6f}')

gain_df = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'gain': baseline_model.booster_.feature_importance(importance_type='gain'),
    'split': baseline_model.booster_.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False).reset_index(drop=True)
gain_df['gain_rank'] = np.arange(1, len(gain_df) + 1)
top5_features = gain_df.head(5)['feature'].tolist()
display(gain_df.head(10))
print('Top-5 gain features:', top5_features)

## 4. Top-5 Feature Ablation

`marginal_contribution = baseline_MAP@12 - MAP@12_after_removal`.

If a feature has high gain but `marginal_contribution <= 0`, it is likely a pseudo-important feature in this validation setting.

In [ ]:
ablation_rows = []
for removed_feature in top5_features:
    ablated_features = [col for col in FEATURE_COLUMNS if col != removed_feature]
    model = train_model(train_df, ablated_features)
    removed_map12 = evaluate_map12(model, valid_features, ablated_features)
    marginal_contribution = baseline_map12 - removed_map12
    gain_row = gain_df[gain_df['feature'] == removed_feature].iloc[0]
    ablation_rows.append({
        'removed_feature': removed_feature,
        'gain_rank': int(gain_row['gain_rank']),
        'gain': float(gain_row['gain']),
        'baseline_map12': baseline_map12,
        'map12_after_removal': removed_map12,
        'marginal_contribution': marginal_contribution,
        'pseudo_important': marginal_contribution <= 0.0,
    })
    print(f'removed={removed_feature:28s} MAP@12={removed_map12:.6f} contribution={marginal_contribution:.6f}')

ablation_df = pd.DataFrame(ablation_rows).sort_values('marginal_contribution', ascending=False)
display(ablation_df)

ablation_path = OUTPUT_DIR / 'lgbm_ablation_top5.csv'
ablation_df.to_csv(ablation_path, index=False)
print('saved:', ablation_path)

pseudo_features = ablation_df.loc[ablation_df['pseudo_important'], 'removed_feature'].tolist()
if pseudo_features:
    print('Potential pseudo-important high-gain features:', pseudo_features)
else:
    print('No high-gain feature has zero/negative marginal contribution in this validation run.')

## 5. SHAP Global and Local Explanation

In [ ]:
shap_n = min(SHAP_SAMPLE_SIZE, valid_features.height)
shap_frame = valid_features.sample(n=shap_n, seed=RANDOM_STATE, shuffle=True)
x_shap = shap_frame.select(FEATURE_COLUMNS).to_pandas()

explainer = shap.TreeExplainer(baseline_model)
shap_explanation = explainer(x_shap)

# LightGBM binary classifiers may return either 2-D values or 3-D class-wise values.
if len(shap_explanation.values.shape) == 3:
    shap_class1 = shap.Explanation(
        values=shap_explanation.values[:, :, 1],
        base_values=shap_explanation.base_values[:, 1],
        data=x_shap,
        feature_names=FEATURE_COLUMNS,
    )
else:
    shap_class1 = shap_explanation

plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_class1, max_display=20, show=False)
plt.tight_layout()
beeswarm_path = OUTPUT_DIR / 'shap_beeswarm.png'
plt.savefig(beeswarm_path, dpi=160, bbox_inches='tight')
plt.show()
print('saved:', beeswarm_path)

plt.figure(figsize=(10, 7))
shap.plots.waterfall(shap_class1[0], max_display=15, show=False)
plt.tight_layout()
waterfall_path = OUTPUT_DIR / 'shap_waterfall_sample0.png'
plt.savefig(waterfall_path, dpi=160, bbox_inches='tight')
plt.show()
print('saved:', waterfall_path)

In [ ]:
mean_abs_shap = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'mean_abs_shap': np.abs(shap_class1.values).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False)
display(mean_abs_shap.head(10))
mean_abs_shap.to_csv(OUTPUT_DIR / 'shap_mean_abs_importance.csv', index=False)

## 6. Short Interpretation

Write your final 2-3 sentence conclusion here after running the notebook.

A typical conclusion format:

- The LightGBM ranker mainly learned to combine baseline candidate priority with recent item popularity and user-item history signals. In the SHAP beeswarm plot, high-ranked candidates and recently popular items tend to push the prediction score upward, while weak or old interaction signals push it downward.
- The ablation table shows whether the gain-based Top-5 features are genuinely useful for MAP@12. If any high-gain feature has zero or negative marginal contribution, it should be treated as pseudo-important because it helps tree splitting but does not improve the final recommendation metric on the validation window.